# Plot the results of synthetic slip inversion based on ANY model of $\mu$ using the synthetic displacement based on ANY model of $\mu$ from a prescribed, ground-truth slip distribution on the fault

* **ANY** means the $\mu$ structure can be either homogeneous or heterogeneous.

* Once we know which elastic property the ground displacement is sensitive to, we may proceed with the inversion.

* The most basic thing to check is if you can recover the ground truth slip and fit the observation under a heterogeneous model of $\mu$. In theory, this should not be different from the homogeneous case, to make sure the code is running properly, you need to pass this test.

* The next level is how different the inferred slip distributions are under homogeneous and heterogeneous cases. The used ground displacement can be from either model, but maybe from heterogeneous $\mu$ structure should be more emphasized.


In [ ]:
# Load libraries
import numpy as np
import pygmt
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import rc
from io import StringIO
import utils as ut

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

In [ ]:
# work dir
datadir = "/home/staff/chao/SSEinv/Nicoya/"

# Define the inversion results path
resultpath = "rst_locking/"
meshname = "nicoya"

# Read the plate interface file
plate_file = "plateinterface/nicoya_slab2_100_10_10.txt"
df_plate = ut.parse_plate_interface_file(datadir + plate_file)
depths = np.array(df_plate['dep'].unique())
# print("depths:", depths)

# Read the plate file in GMT grd format
grd_file = "/home/staff/chao/SSEinv/Nicoya/plateinterface/cam_slab2_dep_02.24.18.grd"

# Read the trench file
trench_file = "plateinterface/trench.gmt.inuse"

# read in the lon and lat of stations
# obs_file = "data/Feng_etal_JGR_2012table1.csv" # original data from Feng et al. 2012
obs_file = "data/Kyriakopoulos_novolcano.csv"    # same as above, but with volcano sites removed

# note that the height is in m, Dt in years, the original displacement data is in mm/yr
# the disp relative to the stable Caribbean plate will be used in the inversion
# From ACOS to VENA are Campaign Sites; From BIJA to VERA are Continuous Sites; From AROL to WARN are Volcano Sites
df = pd.read_csv(datadir + obs_file, sep=",", skiprows=1, \
                   names=['site', 'lat', 'lon', 'height', 'Dt', 'Days', \
                          'vy_ITRF05', 'vy_std_ITRF05', 'vx_ITRF05', 'vx_std_ITRF05', 'vz_ITRF05', 'vz_std_ITRF05', \
                          'vy_Car', 'vy_std_Car', 'vx_Car', 'vx_std_Car', 'vz_Car', 'vz_std_Car'])
df['lon'] = -1*df['lon'] # convert to east positive, as the original data is west positive


In [ ]:
# Decide the weights of the horizontal, vertical components
f_h, f_v = 1, 1/2
# Print the weights of the data
print( "Data weight horizontal / vertical: %.2f / %.2f" %(f_h, f_v) )
# Take the inverse for saving the name of the weights
w_h, w_v = int(1/f_h), int(1/f_v)

# Define the reference point (center of the projection)
lon0, lat0 = -85.5+360, 10
R = 6371.0  # Earth radius in km

# Load the L-curve with $\rho$ fixed 

* 1. "nicoya" mesh -- larger fault interface; use both trench parallel and normal components

* 2. "nicoya" mesh -- larger fault interface; use only trench normal components

* 3. "nicoya2" mesh -- smaller fault interface to roughly match the other studies; use both trench parallel and normal components

* 4. "nicoya2" mesh -- smaller fault interface to roughly match the other studies;use only trench normal components

In [ ]:
# Load the L-curve with rho fixed
# gammas = np.logspace(-2,5,8)
gammas_s = [4e1, 1e2, 2e2, 4e2, 8e2, 1e3, 4e3, 1e4]
rho_s9 = 1e9

meshname1 = "nicoya"  # larger fault interface
meshname2 = "nicoya2"   # This has a smaller fault interface

type1 = "both"
type2 = "dip"

# "nicoya" mesh -- larger fault interface; use both trench parallel and normal components
outFileName11 = f"Lcurve{type1}rs{rho_s9:.0e}{meshname1}.txt"
misfits11 = pd.read_csv(datadir + resultpath + outFileName11, sep=r'\s+',
                     names=['d_mis', 'm_mis'])
# print(misfits11)

# "nicoya" mesh -- larger fault interface; use only trench normal components
outFileName21 = f"Lcurve{type2}rs{rho_s9:.0e}{meshname1}.txt"
misfits21 = pd.read_csv(datadir + resultpath + outFileName21, sep=r'\s+',
                     names=['d_mis', 'm_mis'])
# print(misfits21)

# "nicoya2" mesh -- smaller fault interface to roughly match the other studies; use both trench parallel and normal components
outFileName12 = f"Lcurve{type1}rs{rho_s9:.0e}{meshname2}.txt"
misfits12 = pd.read_csv(datadir + resultpath + outFileName12, sep=r'\s+',
                     names=['d_mis', 'm_mis'])
# print(misfits12)

# "nicoya2" mesh -- smaller fault interface to roughly match the other studies;use only trench normal components
outFileName22 = f"Lcurve{type2}rs{rho_s9:.0e}{meshname2}.txt"
misfits22 = pd.read_csv(datadir + resultpath + outFileName22, sep=r'\s+',
                     names=['d_mis', 'm_mis'])
# print(misfits22)


### L-curve criterion ###
color_L = ['silver', 'firebrick']

# Plot L-curve
fig, axes = plt.subplots(1,2,figsize=(6,6), dpi=300)

ax = axes[0]
# ax.set_xlabel(r"$||\mathbf{Gm} - \mathbf{d}||$", fontsize=6)
ax.set_ylabel(r"$||\mathbf{Lm}||$", fontsize=6)
ax.set_title("Using both trench normal and parallel vel.", fontsize=8)
ax.tick_params(labelsize=8)

ax.plot(misfits11['d_mis'], misfits11['m_mis'], 'o-', color='blue', markerfacecolor=color_L[0],
        linewidth=1.0, markersize=4, label=r"Larger fault; $\rho$ = {:.1e}".format(rho_s9))
ax.plot(misfits12['d_mis'], misfits12['m_mis'], 'o-', color='red', markerfacecolor=color_L[0],
        linewidth=1.0, markersize=4, label=r"Smaller fault; $\rho$ = {:.1e}".format(rho_s9))
for i, gamma in enumerate(gammas_s):
    ax.text(misfits11['d_mis'].iloc[i], misfits11['m_mis'].iloc[i]+0.25, f"{gamma:.0f}", fontsize=6, color='k')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=6)


ax = axes[1]
# ax.set_xlabel(r"$||\mathbf{Gm} - \mathbf{d}||$", fontsize=6)
# ax.set_ylabel(r"$||\mathbf{Lm}||$", fontsize=6)
ax.set_title("Using only trench normal vel.", fontsize=8)
ax.tick_params(labelsize=8)

ax.plot(misfits21['d_mis'], misfits21['m_mis'], 'o-', color='blue', markerfacecolor=color_L[0],
        linewidth=1.0, markersize=4, label=r"Larger fault; $\rho$ = {:.1e}".format(rho_s9))
ax.plot(misfits22['d_mis'], misfits22['m_mis'], 'o-', color='red', markerfacecolor=color_L[0],
        linewidth=1.0, markersize=4, label=r"Smaller fault; $\rho$ = {:.1e}".format(rho_s9))
for i, gamma in enumerate(gammas_s):
    ax.text(misfits21['d_mis'].iloc[i], misfits21['m_mis'].iloc[i]+0.25, f"{gamma:.0f}", fontsize=6, color='k')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=6)




plt.show()